# Talent Marketplace — Data Analysis
**Portfolio project | Carlos Hisamitsu**

End-to-end analysis of a synthetic HR dataset modelled on a real internal talent marketplace:
vendor data ingestion → data quality → workforce analytics → career impact of gig participation.

**Dataset:** 5,000 employees · 1,500 gigs · 3-year window (2023–2025) · 30 countries  
**Data source:** `data/dirty/` — realistic data quality issues injected (see `injection_manifest.csv`)

---


## 1. Setup

In [178]:
# Uncomment to install if needed
# %pip install pandas numpy matplotlib seaborn scipy statsmodels pyarrow openpyxl

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import statsmodels.api as sm

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RAW_DIR    = Path('../data/dirty')
EXPORT_DIR = Path('../data/export')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

LEVELS = ['L1','L2','L3','L4','L5','L6','L7']
REF_DATE = pd.Timestamp('2025-12-31')

print('Ready.')
print(f'Reading from: {RAW_DIR.resolve()}')


Ready.
Reading from: C:\development\data-analytics\hr-analytics-pipeline\data\dirty


## 2. Data Audit

Before transforming anything, profile every file.  
Goal: understand shape, nulls, and duplicates — and flag anything unexpected.


In [179]:
FILES = {
    'employee_master'     : ('raw_employee_master.csv',
                             ['birth_date','hire_date','termination_date']),
    'job_assignment'      : ('raw_employee_job_assignment.csv',
                             ['start_date','end_date']),
    'skill_dim'           : ('dim_skill.csv', []),
    'employee_skills'     : ('raw_employee_skills.csv', ['added_date']),
    'gig_master'          : ('raw_gig_master.csv', ['posted_date']),
    'gig_req_skills'      : ('raw_gig_required_skills.csv', []),
    'applications'        : ('raw_gig_applications_and_assignments.csv',
                             ['application_date','manager_approval_date',
                              'assignment_start_date','assignment_end_date']),
    'activity_log'        : ('raw_user_activity_log.csv', ['event_timestamp']),
    'training_master'     : ('raw_training_master.csv', []),
    'training_skills'     : ('raw_training_skills.csv', []),
    'training_records'    : ('raw_training_records.csv', ['completion_date']),
}

raw = {}
audit = []

for key, (fname, date_cols) in FILES.items():
    df = pd.read_csv(RAW_DIR / fname,
                     parse_dates=date_cols if date_cols else False,
                     low_memory=False)
    raw[key] = df

    null_cols = (df.isnull().sum() / len(df) * 100).round(1)
    null_cols = null_cols[null_cols > 0].sort_values(ascending=False)

    if not null_cols.empty: # null_cols:
        worst_null_col = null_cols.idxmax()
        worst_null_pct = null_cols.max()
        worst_null_list = ", ".join(f"{col}: {round(pct)}%" for col, pct in null_cols.items())
    else:
        worst_null_col = None
        worst_null_pct = 0
        worst_null_list = ""

    audit.append({
        'table'             : key,
        'rows'              : len(df),
        'duplicates'        : int(df.duplicated().sum()),
        'columns'           : df.shape[1],
        'null_cols'         : len(null_cols),
        'worst_null_col'    : worst_null_col,
        'worst_null%'       : round(worst_null_pct, 1),
        "null_list"         : worst_null_list
    })

audit_df = pd.DataFrame(audit)
print(audit_df.to_string(index=False))


           table   rows  duplicates  columns  null_cols         worst_null_col  worst_null%                                                                                                      null_list
 employee_master   5000           0       12          2       termination_date        73.10                                                                   termination_date: 73%, employment_status: 1%
  job_assignment   8678           0       12          2               end_date        59.20                                                                                  end_date: 59%, manager_id: 5%
       skill_dim     80           0        3          0                    NaN         0.00                                                                                                               
 employee_skills  94490        2204        5          1           skill_source         2.90                                                                                               sk

**What I'm looking for here:**
- Duplicates
- Unexpected nulls in key columns (termination dates, manager IDs)
- Row counts consistent with what the source system should export

Issues found will be addressed one by one in Section 3.


## 3. Data Quality

Real data is never clean — especially when it comes from vendor SFTP exports,
HR systems with optional fields, or LMS webhooks that retry on failure.

For each table I run targeted checks against the actual data.
Each check produces a count of affected rows and a sample for inspection.
Issues found are then fixed with a documented, conservative transform.

The approach: **Detect → Inspect → Fix → Verify**


### 3.1 Employee master

In [180]:
emp = raw['employee_master'].copy()
issues = {}

# ── Check 1: Terminated employees missing termination_date ───────────────────
# Logic: status = Terminated implies a date exists. Null means the HR admin
# updated the employment status but skipped the date field.
mask_term_null = (emp['employment_status'].isin(['Terminated', 'Retired'])) & emp['termination_date'].isna()
issues['Terminated or Retired with no termination_date'] = int(mask_term_null.sum())
if mask_term_null.any():
    print(f"[FOUND] Terminated or Retired with no termination_date: {mask_term_null.sum()} rows")
    print(emp[mask_term_null][['employee_id','hire_date','employment_status','termination_date']].head(3).to_string(index=False))
    print()

# Fix: impute using median tenure of employees who do have a date
median_days = int(
    (emp.dropna(subset=['termination_date'])['termination_date'] -
     emp.dropna(subset=['termination_date'])['hire_date']).dt.days.median()
)

emp.loc[mask_term_null, 'termination_date'] = (
    emp.loc[mask_term_null, 'hire_date'] + pd.Timedelta(days=median_days)
)
emp['term_date_imputed'] = mask_term_null
print(f"  Fix: imputed as hire_date + {median_days} days (median tenure of terminated employees)")
print(f"  Verify: nulls remaining = {emp[emp['employment_status'].isin(['Terminated', 'Retired'])]['termination_date'].isna().sum()}")


[FOUND] Terminated or Retired with no termination_date: 49 rows
 employee_id  hire_date employment_status termination_date
EMP-65FAF7BC 2021-03-07        Terminated              NaT
EMP-B44938AE 2018-12-24        Terminated              NaT
EMP-84910D2C 2024-12-16           Retired              NaT

  Fix: imputed as hire_date + 923 days (median tenure of terminated employees)
  Verify: nulls remaining = 0


In [181]:
# ── Check 2: Future hire dates ───────────────────────────────────────────────
# Logic: hire_date cannot be after our analysis date.
# Cause: HR pre-loads upcoming starters before their actual start date.
mask_future = emp['hire_date'] > REF_DATE
issues['Future hire_date'] = int(mask_future.sum())
if mask_future.any():
    print(f"[FOUND] Future hire_date: {mask_future.sum()} rows out of {len(emp)} records")
    print(emp[mask_future][['employee_id','hire_date','employment_status']].head(3).to_string(index=False))
    print()

emp = emp[~mask_future].copy()
print(f"  Fix: drop future hired employees after {REF_DATE.date()}")
print(f"  Verify: future dates remaining = {(emp['hire_date'] > REF_DATE).sum()} / employee records remaining: {len(emp)}")


[FOUND] Future hire_date: 63 rows out of 5000 records
 employee_id  hire_date employment_status
EMP-002CD414 2026-06-14            Active
EMP-BD5C9F30 2026-02-24            Active
EMP-B71B028B 2026-03-08               NaN

  Fix: drop future hired employees after 2025-12-31
  Verify: future dates remaining = 0 / employee records remaining: 4937


In [182]:
# ── Check 3: Missing employment_status ───────────────────────────────────────
# Logic: every employee needs a status. Nulls suggest a middleware integration
# that silently dropped the field on a partial API response.
mask_no_status = emp['employment_status'].isna() | (emp['employment_status'] == '')
issues['Null employment_status'] = int(mask_no_status.sum())
if mask_no_status.any():
    print(f"[FOUND] Null employment_status: {mask_no_status.sum()} rows")
    print(emp[mask_no_status][['employee_id','hire_date','termination_date']].head(3).to_string(index=False))
    print()

emp.loc[mask_no_status & emp['termination_date'].notna(), 'employment_status'] = 'Terminated'
emp.loc[mask_no_status & emp['termination_date'].isna(),  'employment_status'] = 'Active'

emp['employment_status_imputed'] = mask_no_status
print(f"  Fix: derived from termination_date for {mask_no_status.sum()} rows")
print(f"  Verify: nulls remaining = {emp['employment_status'].isna().sum()}")


[FOUND] Null employment_status: 49 rows
 employee_id  hire_date termination_date
EMP-112B1E91 2019-11-08              NaT
EMP-EB9C11B2 2020-03-01              NaT
EMP-2489099C 2021-03-25              NaT

  Fix: derived from termination_date for 49 rows
  Verify: nulls remaining = 0


In [183]:
# ── Check 4: Duplicate email addresses ───────────────────────────────────────
# Logic: company_email must be unique. Duplicates appear when an employee is re-hired
# with same old email or another employee with same name of a former employee is hired.
# (duplicated active emails are blocked at the AD creation, so we will not consider this case)
mask_dup = emp.duplicated(subset=['company_email'], keep=False)
issues['Duplicate company_email'] = int(mask_dup.sum())
if mask_dup.any():
     print(f"\n[FOUND] Duplicate emails: {mask_dup.sum()} rows across {emp[mask_dup]['company_email'].nunique()} addresses")
     print(emp[mask_dup].groupby('company_email').head(2)[['employee_id','first_name','last_name','company_email']].head(4).to_string(index=False))
     print()

emp_term = emp['employment_status'].isin(['Terminated', 'Retired'])
emp.loc[emp_term, 'company_email'] = (
    emp.loc[emp_term, 'employee_id'].astype(str) + '@' + 
    emp.loc[emp_term, 'company_email'].str.split('@').str[1]
)
print("  Fix: use employee_id for terminated or retired employees' emails")

still_dup = emp.duplicated(subset=['company_email'], keep=False)
issues['Email replaced (data error)'] = int(still_dup.sum())
if still_dup.any():
     print(f"\n[FOUND] There are still duplicate emails: {still_dup.sum()} rows across {emp[still_dup]['company_email'].nunique()} addresses")
     print(emp[still_dup].groupby('company_email').head(2)[['employee_id','first_name','last_name','company_email']].head(4).to_string(index=False))
     print()
     emp.loc[still_dup, 'company_email'] = (
          emp.loc[still_dup, 'employee_id'].astype(str) + '@' + 
          emp.loc[still_dup, 'company_email'].str.split('@').str[1]
          )
     
emp['duplicated_email_imputed'] = still_dup
print("  Fix: use employee_id for emails that are still duplicated after fixing terminated/retired employees")
print(f"  Verify: duplicates remaining = {emp.duplicated(subset=['company_email']).sum()}")



[FOUND] Duplicate emails: 143 rows across 71 addresses
 employee_id first_name last_name                company_email
EMP-56A30F13      Piotr     Meier     laura.hansen@example.com
EMP-C169FD54       Lena  Hoffmann  lena.hoffmann.2@example.com
EMP-7CCAD74F   Isabella    Miller  isabella.miller@example.com
EMP-68406935     Werner     Petit franziska.thomas@example.com



  Fix: use employee_id for terminated or retired employees' emails

[FOUND] There are still duplicate emails: 62 rows across 31 addresses
 employee_id first_name last_name                company_email
EMP-56A30F13      Piotr     Meier     laura.hansen@example.com
EMP-7CCAD74F   Isabella    Miller  isabella.miller@example.com
EMP-68406935     Werner     Petit franziska.thomas@example.com
EMP-2B4257B7     Monika  Goossens  monika.goossens@example.com

  Fix: use employee_id for emails that are still duplicated after fixing terminated/retired employees
  Verify: duplicates remaining = 0


In [184]:
# ── Check 5: Name encoding anomalies ─────────────────────────────────────────
# Logic: flag names with characters suggesting a UTF-8/Latin-1 encoding mismatch.
# Common in vendor SFTP exports where the server locale is misconfigured.
# We flag only — auto-correcting names without a reference list risks data loss.
emp['name_encoding_flag'] = (
    emp['first_name'].apply(lambda x: isinstance(x, str) and not x.isascii()) |
    emp['last_name'].apply(lambda x: isinstance(x, str) and not x.isascii())
)
issues['Name encoding anomalies (flagged)'] = int(emp['name_encoding_flag'].sum())
if emp['name_encoding_flag'].any():
    print(f"\n[FOUND] Encoding anomalies: {emp['name_encoding_flag'].sum()} rows — flagged for manual review")
    print(emp[emp['name_encoding_flag']][['employee_id','first_name','last_name']].head(4).to_string(index=False))



[FOUND] Encoding anomalies: 675 rows — flagged for manual review
 employee_id first_name last_name
EMP-FE18136A       Sven   Sánchez
EMP-D9167930      Sofia    García
EMP-D9B91188     Carlos     López
EMP-C778CAC4     Amélie    Schulz


In [185]:
print("\n=== Employee master — summary ===")
for k, v in issues.items():
    print(f"  {'FOUND':<6s} | {k:<47s} | {v:>5,} rows")



=== Employee master — summary ===
  FOUND  | Terminated or Retired with no termination_date  |    49 rows
  FOUND  | Future hire_date                                |    63 rows
  FOUND  | Null employment_status                          |    49 rows
  FOUND  | Duplicate company_email                         |   143 rows
  FOUND  | Email replaced (data error)                     |    62 rows
  FOUND  | Name encoding anomalies (flagged)               |   675 rows


### 3.2 Job assignments (SCD2)

In [186]:
ja = raw['job_assignment'].copy()
ja['job_level'] = pd.Categorical(ja['job_level'], categories=LEVELS, ordered=True)
ja_issues = {}

# ── Check 1: Employee is their own manager ───────────────────────────────────
# Logic: manager_id should never equal employee_id.
# Cause: SuccessFactors/HRIS-HCM sets manager_id = employee_id for org chart root nodes
# (e.g. CEO, country GMs) when no superior position exists in the system.
mask_self = ja['manager_id'] == ja['employee_id']
ja_issues['Self-referencing manager'] = int(mask_self.sum())
if mask_self.any():
    print(f"[FOUND] Self-referencing manager_id: {mask_self.sum()} rows")
    print(ja[mask_self][['employee_id','manager_id','job_level','function']].head(3).to_string(index=False))
    print()
ja.loc[mask_self, 'manager_id'] = None
print(f"  Fix: set to null — no manager is more accurate than a self-reference")

# ── Check 2: Missing manager_id ───────────────────────────────────────────────
# Logic: most employees should have a manager. High null rates typically
# indicate matrix org relationships not mapped in SuccessFactors/HRIS-HCM position management.
mask_no_mgr = ja['manager_id'].isna() | (ja['manager_id'] == '')
ja_issues['Missing manager_id'] = int(mask_no_mgr.sum())
print(f"\n[FOUND] Missing manager_id: {mask_no_mgr.sum()} rows ({mask_no_mgr.mean()*100:.1f}% of assignments)")
ja['manager_missing'] = mask_no_mgr
print(f"  Action: flagged — cannot impute manager without org chart data")


[FOUND] Self-referencing manager_id: 42 rows
 employee_id   manager_id job_level    function
EMP-3F41E93C EMP-3F41E93C        L6  Regulatory
EMP-295BC9DE EMP-295BC9DE        L2 Procurement
EMP-E916B745 EMP-E916B745        L2          IT

  Fix: set to null — no manager is more accurate than a self-reference

[FOUND] Missing manager_id: 475 rows (5.5% of assignments)
  Action: flagged — cannot impute manager without org chart data


In [187]:
# ── Check 3: Overlapping assignment periods ───────────────────────────────────
# Logic: consecutive assignment periods for the same employee must not overlap.
# Cause: two records created during a system migration, both marked active.
ja_tmp = ja.sort_values(['employee_id','start_date'])
ja_tmp['prev_end'] = pd.to_datetime(
    ja_tmp.groupby('employee_id')['end_date'].shift(1), errors='coerce')
overlap_mask = (ja_tmp['prev_end'].notna() &
                ja_tmp['start_date'].notna() &
                (ja_tmp['start_date'] < ja_tmp['prev_end']))
ja_issues['Overlapping periods'] = int(overlap_mask.sum())
if overlap_mask.any():
    print(f"[FOUND] Overlapping assignment periods: {overlap_mask.sum()} rows")
    print(ja_tmp[overlap_mask][['employee_id','start_date','end_date','prev_end']].head(3).to_string(index=False))
    print()
    ja.loc[overlap_mask[overlap_mask].index, 'start_date'] = ja_tmp.loc[overlap_mask,'prev_end'].values
    print(f"  Fix: adjusted start_date to align with previous end_date")


[FOUND] Overlapping assignment periods: 28 rows
 employee_id start_date   end_date   prev_end
EMP-11FA09BE 2024-07-29 2024-12-04 2024-08-25
EMP-13BD9955 2019-11-18        NaT 2019-12-04
EMP-21350978 2025-11-10        NaT 2025-11-24

  Fix: adjusted start_date to align with previous end_date


In [188]:
# ── Check 4: Spurious end_date on most recent assignment ─────────────────────
# Logic: the latest assignment per employee should be open (end_date = null).
# Except for Terminated or Retired employees.
# Cause: migration script incorrectly closed all records including current ones.
ja_s = ja.sort_values(['employee_id','start_date'])
is_latest = ja_s.groupby('employee_id')['start_date'].transform('max') == ja_s['start_date']

terminated_ids = set(
    emp.loc[emp['employment_status'].isin(['Terminated','Retired']), 'employee_id']
)
spurious_end = (
    is_latest &
    ja_s['end_date'].notna() &
    ~ja_s['employee_id'].isin(terminated_ids)
)
ja_issues['Spurious end_date on latest assignment'] = int(spurious_end.sum())
if spurious_end.any():
    print(f"[FOUND] Latest assignment with end_date: {spurious_end.sum()} rows")
    print(ja_s[spurious_end][['employee_id','start_date','end_date','job_level']].head(3).to_string(index=False))
    print()
ja.loc[spurious_end[spurious_end].index, 'end_date'] = None
print(f"  Fix: cleared end_date for {spurious_end.sum()} latest assignments")

[FOUND] Latest assignment with end_date: 69 rows
 employee_id start_date   end_date job_level
EMP-01A4023E 2023-03-20 2023-10-04        L5
EMP-01D9429F 2025-11-27 2026-02-06        L5
EMP-03EC2C2D 2025-07-12 2026-05-03        L3

  Fix: cleared end_date for 69 latest assignments


In [190]:
# ── Consistency: Terminated/Retired employees with open assignments ────────────
# Cause: HR system and job assignment module fell out of sync — status updated
# in employee master but the assignment was never closed.
open_for_leavers = (
    is_latest &
    ja_s['end_date'].isna() &
    ja_s['employee_id'].isin(terminated_ids)
)

if open_for_leavers.any():
    affected_term_dates = (
        ja_s[open_for_leavers][['employee_id']]
        .merge(emp[['employee_id','termination_date','term_date_imputed']], 
               on='employee_id', how='left')
    )
    ja.loc[open_for_leavers[open_for_leavers].index, 'end_date'] = \
        affected_term_dates['termination_date'].values
    ja.loc[open_for_leavers[open_for_leavers].index, 'end_date_imputed'] = \
        affected_term_dates['term_date_imputed'].values

print(f"[FOUND] Open assignments for Terminated/Retired: {open_for_leavers.sum()} — closed using termination_date")
imputed_count = affected_term_dates['term_date_imputed'].sum() if open_for_leavers.any() else 0
print(f"  - of which {imputed_count} used an imputed termination_date (flagged)")


[FOUND] Open assignments for Terminated/Retired: 1407 — closed using termination_date
  - of which 54 used an imputed termination_date (flagged)


In [191]:
# ── Derive promotion events and current flag on fully corrected data ──────────
ja = ja.sort_values(['employee_id','start_date'])
ja['prev_level']   = ja.groupby('employee_id')['job_level'].shift(1)
ja['is_promotion'] = ja['prev_level'].notna() & (ja['job_level'] > ja['prev_level'])
ja['is_current']   = ja['end_date'].isna()

print(f"\n=== Job assignments — summary ===")
for k, v in ja_issues.items():
    print(f"  {'FOUND':<6s} | {k:<45s} | {v:>5,} rows")
print(f"\nPromotion events detected: {ja['is_promotion'].sum():,}")
print(f"Open assignments: {ja['is_current'].sum():,}")


=== Job assignments — summary ===
  FOUND  | Self-referencing manager                      |    42 rows
  FOUND  | Missing manager_id                            |   475 rows
  FOUND  | Overlapping periods                           |    28 rows
  FOUND  | Spurious end_date on latest assignment        |    69 rows

Promotion events detected: 393
Open assignments: 3,797
